# 00 — Setup

## What this demonstrates

This notebook establishes the Data Science Workbench as a registered consumer of the Verity governance platform. After running it, every subsequent notebook's runtime calls, execution contexts, and decision-log rows are cleanly attributed to the `ds_workbench` application — which makes them trivially separable from any other application's activity, and trivially cleanable via `99_cleanup.ipynb`.

**Verity capabilities exercised**

- Reporting dashboard counts (`GET /api/v1/reporting/dashboard-counts`).
- Application lookup, registration, and entity mapping (`/api/v1/applications/*`).
- Reading a resolved agent config to seed a typical workbench interaction (`GET /api/v1/agents/{name}/config`).


## Prerequisites

Only one: the Verity service must be reachable at `VERITY_API_URL`. The helper reads the env var on construction and falls back to `http://localhost:8000` — so inside Docker JupyterLab `VERITY_API_URL=http://verity:8000` is set by compose, and from VSCode on host the default localhost URL works.


In [ ]:
import os, sys
HERE = os.getcwd()
if os.path.basename(HERE) != 'ds_workbench':
    for candidate in (os.path.dirname(HERE),
                      os.path.dirname(os.path.dirname(HERE)),
                      '/home/jovyan/work'):
        if os.path.isdir(os.path.join(candidate, 'utility')):
            sys.path.insert(0, candidate); break

from utility.verity import VerityAPI, VerityAPIError
from utility.html import inject_style, badge, render_list, render_detail, render_cards

inject_style()   # apply Verity-UI styles to all subsequent cell outputs

In [ ]:
# Open a client. VERITY_API_URL env var decides the target;
# the default lands on localhost for VSCode-on-host users.
v = VerityAPI(application='ds_workbench')
print(f'base_url            = {v.base_url}')
print(f'default application = {v.application}')

In [ ]:
# Health + dashboard counts — fails fast if Verity is unreachable.
counts = v.dashboard_counts()
render_cards([
    ('Agents',         counts['agent_count'],      None),
    ('Tasks',          counts['task_count'],       None),
    ('Pipelines',      counts['pipeline_count'],   None),
    ('Prompts',        counts['prompt_count'],     None),
    ('Tools',          counts['tool_count'],       None),
    ('Inference cfg',  counts['config_count'],     None),
    ('Decisions',      counts['total_decisions'],  'total'),
    ('Overrides',      counts['total_overrides'],  'logged'),
    ('Open incidents', counts['open_incidents'],   ''),
])

## Execute

Idempotent registration of the `ds_workbench` application. If it already exists we reuse it; otherwise a fresh application row is created.


In [ ]:
app = v.ensure_application_registered(
    name='ds_workbench',
    display_name='Data Science Workbench',
    description='Interactive Verity capability walkthrough notebooks.',
)
render_detail(
    'ds_workbench application',
    subtitle=app['name'],
    sections=[
        {'title': 'Identity', 'fields': [
            ('Name',         app.get('name')),
            ('Display name', app.get('display_name')),
            ('Description',  app.get('description')),
            ('ID',           app.get('id')),
            ('Created',      app.get('created_at')),
        ]},
    ],
)

## Review results

The Verity catalog as it exists right now, plus the activity footprint our workbench has accumulated so far (fresh install should be zero decisions, zero mappings).


In [ ]:
applications = v.list_applications()
render_list(
    applications,
    columns=[('name','Name'), ('display_name','Display'), ('description','Description')],
    title='Registered applications',
    description='Every business app that consumes Verity is registered here.',
)

In [ ]:
activity = v.get_app_activity()
render_cards([
    ('Decisions logged',     activity['decision_count'],           'this app'),
    ('Overrides recorded',   activity['override_count'],           'this app'),
    ('Execution contexts',   activity['execution_context_count'],  'this app'),
    ('Entity mappings',      activity['entity_mapping_count'],     'this app'),
])

In [ ]:
# Sanity check: we can resolve an agent's full config. This is
# the blob subsequent notebooks rely on — header + inference
# config + prompt assignments + tool authorizations.
config = v.get_agent_config('triage_agent')
render_detail(
    config.get('agent_name', 'agent'),
    subtitle=f"v{config.get('version_label', '?')}",
    header_badges=[
        (config.get('lifecycle_state','?'), config.get('lifecycle_state','')),
        (config.get('materiality_tier','?'), config.get('materiality_tier','')),
    ],
    sections=[
        {'title': 'Inference config', 'fields': [
            ('Name',        config['inference_config'].get('name')),
            ('Model',       config['inference_config'].get('model_name')),
            ('Temperature', config['inference_config'].get('temperature')),
            ('Max tokens',  config['inference_config'].get('max_tokens')),
        ]},
        {'title': f"Prompts ({len(config.get('prompts') or [])})",
         'table': {
             'columns': [
                 ('prompt_name','Prompt'),
                 ('version_number','Version'),
                 ('api_role','Role','neutral'),
                 ('governance_tier','Tier'),
                 ('execution_order','Order'),
             ],
             'rows': config.get('prompts') or [],
         }},
        {'title': f"Tools ({len(config.get('tools') or [])})",
         'table': {
             'columns': [
                 ('tool_name','Tool'),
                 ('transport','Transport','neutral'),
                 ('mcp_server_name','MCP server'),
             ],
             'rows': config.get('tools') or [],
         }},
    ],
)

---

Setup complete. Move on to the component folders under `notebooks/` to exercise individual Verity capabilities. When you're done for the session (or want to start clean), run **`99_cleanup.ipynb`** to purge activity and unregister this workbench.
